# Resolución orientativa de la práctica de la semana 2
120 productos sintéticos en seis categorías. Identificador y categoría se conservan para inspección; solo la descripción se vectoriza. La repetición deliberada de expresiones permite practicar duplicados y empates. La categoría es una referencia didáctica, no una evaluación de preferencias reales.
Ejecuta desde la carpeta Recursos. La propuesta es una solución posible; compara configuraciones y justifica tus decisiones.

## Datos incluidos en el propio notebook
Ejecuta las celdas en orden dentro de una carpeta de trabajo. La primera genera los CSV sintéticos; no necesitas descargar datos. Puedes conservar esa celda sin ejecutar si quieres analizar tus propias copias de los archivos.

In [1]:
# Generación de los dos conjuntos sintéticos de la práctica.
import numpy as np
import pandas as pd
plantillas = [
    ("telefonia", "telefono celular smartphone pantalla bateria camara"),
    ("audio", "audifonos auriculares bluetooth sonido microfono"),
    ("computacion", "computador portatil laptop memoria teclado pantalla"),
    ("cocina", "licuadora cocina vaso potencia cuchillas"),
    ("deporte", "zapatillas calzado correr deporte ligero"),
    ("hogar", "lampara escritorio luz lectura regulable")]
filas = []
extras = ["compacto", "recargable", "resistente", "portatil", "nuevo"]
for i in range(120):
    categoria, descripcion = plantillas[i % 6]
    filas.append([f"P{i+1:03}", f"{categoria.capitalize()} modelo {i+1}",
                  categoria, descripcion + " " + extras[i % 5]])
columnas = ["producto_id", "nombre", "categoria", "descripcion"]
pd.DataFrame(filas, columns=columnas).to_csv("catalogo_productos.csv", index=False)
rng = np.random.default_rng(42)
transacciones = pd.DataFrame({
    "transaccion_id": [f"T{i:03d}" for i in range(1, 201)],
    "monto_usd": rng.lognormal(3.6, .45, 200).round(2),
    "compras_24h": rng.poisson(2, 200) + 1,
    "antiguedad_cuenta_dias": rng.integers(30, 1000, 200)})
# Combinaciones extremas didácticas, sin etiquetas de fraude.
transacciones.loc[192:199, "monto_usd"] = np.arange(500, 900, 50)
transacciones.loc[192:199, "compras_24h"] = np.arange(15, 23)
transacciones.loc[192:199, "antiguedad_cuenta_dias"] = np.arange(1, 9)
transacciones.to_csv("transacciones_sinteticas.csv", index=False)


In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
df = pd.read_csv("catalogo_productos.csv")
assert df.producto_id.is_unique and df.descripcion.notna().all()
vectorizador = TfidfVectorizer(lowercase=True,strip_accents="unicode",ngram_range=(1,2))
X = vectorizador.fit_transform(df.descripcion)
S = cosine_similarity(X)
no_nulos = np.asarray(X.getnnz(axis=1)).ravel() > 0
assert np.allclose(np.diag(S)[no_nulos],1.0,atol=1e-10)
def recomendar(idx,k=3):
    puntajes = S[idx].copy()
    puntajes[idx] = -np.inf  # Excluir por índice, incluso si hay empates.
    orden = np.argsort(-puntajes,kind="stable")[:k]
    resultado = df.iloc[orden][["producto_id","nombre","categoria","descripcion"]].copy()
    resultado["similitud"] = puntajes[orden]
    return resultado
for idx in [0,1]:
    print("Consulta:",df.iloc[idx]["nombre"])
    print(recomendar(idx).to_string(index=False))
consulta = vectorizador.transform(["auriculares para escuchar musica"])
print("Consulta sin términos conocidos:",consulta.nnz==0)
print("Nota: coincidencia textual no demuestra relevancia para usuarios reales.")

# Contraste básico con unigramas sobre las mismas consultas.
vectorizador_uni=TfidfVectorizer(lowercase=True,strip_accents="unicode")
U=vectorizador_uni.fit_transform(df.descripcion)
Su=cosine_similarity(U)
for idx in [0,1]:
    puntajes=Su[idx].copy();puntajes[idx]=-np.inf
    top=np.argsort(-puntajes,kind="stable")[:3]
    print("Unigramas, consulta",df.iloc[idx].producto_id,":",list(df.iloc[top].producto_id))
# Demostración explícita de consulta fuera del vocabulario.
assert vectorizador.transform(["xyzterminodesconocido"]).nnz==0
print("Consulta vacía o fuera del vocabulario: no se presentan resultados como relevantes.")


Consulta: Telefonia modelo 1
producto_id              nombre categoria                                                  descripcion  similitud
       P031 Telefonia modelo 31 telefonia telefono celular smartphone pantalla bateria camara compacto        1.0
       P061 Telefonia modelo 61 telefonia telefono celular smartphone pantalla bateria camara compacto        1.0
       P091 Telefonia modelo 91 telefonia telefono celular smartphone pantalla bateria camara compacto        1.0
Consulta: Audio modelo 2
producto_id          nombre categoria                                                 descripcion  similitud
       P032 Audio modelo 32     audio audifonos auriculares bluetooth sonido microfono recargable        1.0
       P062 Audio modelo 62     audio audifonos auriculares bluetooth sonido microfono recargable        1.0
       P092 Audio modelo 92     audio audifonos auriculares bluetooth sonido microfono recargable        1.0
Consulta sin términos conocidos: False
Nota: coinciden

## Interpretación orientativa
Los empates son deliberados: varios productos comparten la misma descripción. Excluir por identificador evita que el producto consultado se recomiende a sí mismo incluso cuando la similitud vale uno para varias filas. El desempate estable hace reproducible el orden, pero no lo convierte en una preferencia de los compradores.

La comparación entre unigramas y bigramas puede producir recuperaciones muy parecidas porque el catálogo sintético usa vocabulario repetido por categoría. Eso permite verificar el flujo, pero limita la dificultad del ejemplo. Para evaluar utilidad real, convendría reunir consultas variadas, pedir juicios de relevancia independientes y comparar precisión en los primeros resultados sobre consultas reservadas.

Una consulta sin términos conocidos necesita una respuesta definida, por ejemplo informar que no hay coincidencias y solicitar otra descripción. En un catálogo mayor habría que añadir pruebas con negaciones, atributos incompatibles, textos vacíos y cambios de vocabulario. La categoría del generador sirve para inspección; no reemplaza una evaluación humana ni demuestra impacto comercial.


## Complemento breve de Isolation Forest
Carga las 200 transacciones sintéticas y utiliza únicamente monto_usd, compras_24h y antiguedad_cuenta_dias. Ajusta una configuración fija de Isolation Forest. Presenta las cinco observaciones más inusuales y comenta en 3–5 frases dos de ellas frente a las medianas del lote. El identificador sirve para consultar cada registro. Alcance orientativo: 15 minutos, sin búsqueda de parámetros ni clasificación de fraude.

In [3]:
# Complemento breve: Isolation Forest sobre tres variables numéricas.
from sklearn.ensemble import IsolationForest
transacciones = pd.read_csv("transacciones_sinteticas.csv")
variables_if = ["monto_usd", "compras_24h", "antiguedad_cuenta_dias"]
assert transacciones.transaccion_id.is_unique
assert not transacciones[variables_if].isna().any().any()
# Exploración descriptiva de un lote, sin evaluación de generalización ni etiquetas.
X_if = transacciones[variables_if]
modelo_if = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
modelo_if.fit(X_if)
resultado_if = transacciones.copy()
# score_samples es menor para casos más anómalos; cambiamos el signo al presentar.
resultado_if["puntaje_anomalia"] = -modelo_if.score_samples(X_if)
resultado_if["alerta"] = modelo_if.predict(X_if) == -1
resultado_if = resultado_if.sort_values("puntaje_anomalia", ascending=False, kind="stable")
print("Tabla P2. Cinco transacciones con mayor puntuación de anomalía")
print(resultado_if.head(5).round(4).to_string(index=False))
print("Fuente: elaboración propia con transacciones_sinteticas.csv; semilla 42.")
print("Observaciones:", len(resultado_if), "Alertas:", int(resultado_if.alerta.sum()))
print("Referencia del lote (medianas):", X_if.median().to_dict())
print("contamination=0.05 fija un umbral didáctico; no estima la proporción de fraude.")
print("Las alertas requieren revisión. Sin casos confirmados no se calcula exactitud.")
resultado_if.to_csv("resultado_isolation_forest.csv", index=False)


Tabla P2. Cinco transacciones con mayor puntuación de anomalía
transaccion_id  monto_usd  compras_24h  antiguedad_cuenta_dias  puntaje_anomalia  alerta
          T200      850.0           22                       8            0.7620    True
          T199      800.0           21                       7            0.7078    True
          T193      500.0           15                       1            0.7077    True
          T198      750.0           20                       6            0.6860    True
          T197      700.0           19                       5            0.6767    True
Fuente: elaboración propia con transacciones_sinteticas.csv; semilla 42.
Observaciones: 200 Alertas: 10
Referencia del lote (medianas): {'monto_usd': 37.33, 'compras_24h': 3.0, 'antiguedad_cuenta_dias': 520.5}
contamination=0.05 fija un umbral didáctico; no estima la proporción de fraude.
Las alertas requieren revisión. Sin casos confirmados no se calcula exactitud.


## Solución orientativa del complemento
Complemento orientativo: Isolation Forest marca diez de las 200 transacciones con el umbral fijado. T200 combina 850 dólares, 22 compras en 24 horas y ocho días de antigüedad; T193 combina 500 dólares, quince compras y un día. Las medianas del lote son 37,33 dólares, tres compras y 520,5 días. Esas diferencias ayudan a describir la rareza de los registros, pero no explican causalmente la puntuación ni confirman fraude. Corresponde revisar el historial y el contexto antes de actuar. Sin etiquetas confirmadas, la demostración evidencia detección de anomalías y no capacidad predictiva frente a fraude real.